# 1.2 交叉熵、最大似然与困惑度 / Cross-entropy, maximum likelihood and perplexity

三个名字，同一件事。弄清它们之间只差一个负号、一个平均和一个指数。

Three names for one thing. They differ by a minus sign, an average and an exponential.

讲解与推导见网站对应模块。每个练习先写代码，再运行紧随其后的检查单元。

The explanation and derivations are on the website. For each exercise, write the code, then run the check cell right after it.

## 1. 手写交叉熵与困惑度 / Cross-entropy and perplexity by hand

不调用 `F.cross_entropy`，实现交叉熵、困惑度和每 token 比特数。测试会与 PyTorch 官方实现对比，并检查两个极端情形，即均匀分布的困惑度等于词表大小、完全确定且正确的预测困惑度为 1。

Without calling `F.cross_entropy`, implement cross-entropy, perplexity and bits per token. The tests compare against the PyTorch implementation and check two extremes. A uniform distribution has perplexity equal to the vocabulary size, and a certain, correct prediction has perplexity 1.

In [ ]:
import torch


def cross_entropy(logits, targets):
    """Mean negative log-likelihood in nats.

    logits:  float tensor (N, V), unnormalised scores
    targets: long tensor (N,), the index of the correct token
    Do not call F.cross_entropy, F.nll_loss or F.log_softmax.
    torch.logsumexp is allowed here (you will write it yourself in module 1.3).
    """
    # TODO 1: log-probabilities  log p = logits - logsumexp(logits)
    # TODO 2: pick log p of the correct token for every row
    # TODO 3: return the mean of the negative values
    raise NotImplementedError


def perplexity(logits, targets):
    """Perplexity = exp(mean NLL). Returns a 0-dim tensor."""
    # TODO 4
    raise NotImplementedError


def bits_per_token(logits, targets):
    """The same loss measured in bits instead of nats."""
    # TODO 5
    raise NotImplementedError

In [ ]:
# 检查 / checks
import math
import torch
import torch.nn.functional as F

torch.manual_seed(0)
N, V = 64, 50
logits = torch.randn(N, V, dtype=torch.float64) * 3
targets = torch.randint(0, V, (N,))

ref = F.cross_entropy(logits, targets)
assert torch.allclose(cross_entropy(logits, targets), ref, atol=1e-10), "does not match F.cross_entropy"
assert torch.allclose(perplexity(logits, targets), ref.exp(), atol=1e-8), "perplexity should be exp(cross-entropy)"
assert torch.allclose(bits_per_token(logits, targets), ref / math.log(2), atol=1e-10), "bits = nats / ln 2"

# A model that knows nothing (uniform) has loss ln V and perplexity V.
uniform = torch.zeros(N, V, dtype=torch.float64)
assert abs(cross_entropy(uniform, targets).item() - math.log(V)) < 1e-10
assert abs(perplexity(uniform, targets).item() - V) < 1e-8

# A model that is certain and right has loss 0 and perplexity 1.
sure = torch.full((N, V), -50.0, dtype=torch.float64)
sure[torch.arange(N), targets] = 50.0
assert perplexity(sure, targets).item() < 1 + 1e-8

# Large logits must not overflow.
big = logits * 1e4
assert torch.isfinite(cross_entropy(big, targets)), "overflow: work in log space"

# Gradients flow through your implementation.
x = logits.clone().requires_grad_(True)
cross_entropy(x, targets).backward()
y = logits.clone().requires_grad_(True)
F.cross_entropy(y, targets).backward()
assert torch.allclose(x.grad, y.grad, atol=1e-10)
print("m12_cross_entropy: all tests passed")

<details><summary><b>提示 1 / Hint 1</b></summary>

先得到对数概率 `logits - torch.logsumexp(logits, dim=-1, keepdim=True)`，不要先算概率再取 log。

First get log-probabilities with `logits - torch.logsumexp(logits, dim=-1, keepdim=True)`. Do not compute probabilities and then take the log.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

用 `log_probs[torch.arange(N), targets]` 一次取出每一行里正确 token 的对数概率。

Use `log_probs[torch.arange(N), targets]` to pick the correct token's log-probability in every row at once.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
import math
import torch


def cross_entropy(logits, targets):
    log_probs = logits - torch.logsumexp(logits, dim=-1, keepdim=True)   # (N, V)
    picked = log_probs[torch.arange(logits.shape[0]), targets]           # (N,)
    return -picked.mean()


def perplexity(logits, targets):
    return torch.exp(cross_entropy(logits, targets))


def bits_per_token(logits, targets):
    return cross_entropy(logits, targets) / math.log(2)
```

</details>

## 2. 计算 GPT-2 在一段文本上的困惑度 / Perplexity of GPT-2 on a piece of text

加载 GPT-2 small（约 500 MB，CPU 可运行），自己完成“错位一格”的对齐并计算每个 token 的负对数似然。测试会与 Hugging Face 传入 `labels` 时返回的 loss 对比，并验证把词序打乱后困惑度大幅上升。

Load GPT-2 small (about 500 MB, runs on CPU), do the shift-by-one alignment yourself and compute the negative log-likelihood of every token. The tests compare with the loss Hugging Face returns when `labels` is passed, and check that shuffling the words raises the perplexity sharply.

需要联网下载模型。/ Downloads a model.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


@torch.no_grad()
def token_nlls(model, input_ids):
    """Negative log-likelihood of every token given the tokens before it.

    input_ids: long tensor (1, T)
    returns:   float tensor (T-1,), entry t-1 is  -log p(x_t | x_<t)  for t = 1..T-1
    The first token has no context, so it gets no score.
    """
    # TODO 1: logits = model(input_ids).logits            -> (1, T, V)
    # TODO 2: the logits at position t predict token t+1. Align the two.
    # TODO 3: log-softmax over the vocabulary, pick the true next token, negate
    raise NotImplementedError


def text_perplexity(model, tokenizer, text):
    """Perplexity of `text` under `model`, as a Python float."""
    # TODO 4: tokenise, call token_nlls, return exp(mean)
    raise NotImplementedError

In [ ]:
# 检查 / checks
import random
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained("openai-community/gpt2")
gpt2 = AutoModelForCausalLM.from_pretrained("openai-community/gpt2").eval()

text = "The capital of France is Paris, and the capital of Germany is Berlin."
ids = tok(text, return_tensors="pt").input_ids

nll = token_nlls(gpt2, ids)
assert nll.shape == (ids.shape[1] - 1,), f"expected {ids.shape[1] - 1} scores, got {tuple(nll.shape)}"
assert (nll > 0).all(), "a negative log-likelihood cannot be negative"

# Hugging Face computes the same mean when labels are passed.
with torch.no_grad():
    ref = gpt2(ids, labels=ids).loss
assert torch.allclose(nll.mean(), ref, atol=1e-4), f"mean NLL {nll.mean():.4f} vs reference {ref:.4f}"

ppl = text_perplexity(gpt2, tok, text)
assert abs(ppl - ref.exp().item()) < 1e-2

# Shuffling the words destroys the structure the model relies on.
words = text.split()
random.Random(0).shuffle(words)
ppl_shuffled = text_perplexity(gpt2, tok, " ".join(words))
assert ppl_shuffled > 3 * ppl, "shuffled text should be far more surprising"
print(f"m12_gpt2_ppl: all tests passed (PPL {ppl:.1f}, shuffled {ppl_shuffled:.1f})")

<details><summary><b>提示 1 / Hint 1</b></summary>

位置 $t$ 的 logits 预测的是第 $t+1$ 个 token。所以预测用 `logits[0, :-1]`，目标用 `input_ids[0, 1:]`，两者长度都是 $T-1$。

The logits at position $t$ predict token $t+1$. Use `logits[0, :-1]` as predictions and `input_ids[0, 1:]` as targets. Both have length $T-1$.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

最后一个位置的 logits 没有对应的目标，第一个 token 没有上文，所以都不计入。

The logits at the last position have no target and the first token has no context, so neither is scored.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


@torch.no_grad()
def token_nlls(model, input_ids):
    logits = model(input_ids).logits                  # (1, T, V)
    pred = logits[0, :-1]                             # positions 0..T-2 predict tokens 1..T-1
    gold = input_ids[0, 1:]                           # (T-1,)
    log_probs = torch.log_softmax(pred, dim=-1)
    return -log_probs[torch.arange(gold.shape[0]), gold]


def text_perplexity(model, tokenizer, text):
    input_ids = tokenizer(text, return_tensors="pt").input_ids
    return torch.exp(token_nlls(model, input_ids).mean()).item()
```

</details>

## 3. 推导最小化交叉熵等价于最大似然 / Derive that minimizing cross-entropy is maximum likelihood

合上上面的推导，在纸上完成下面四问。
(a) 证明最大化 $\prod_i p_\theta(x^{(i)})$ 与最小化平均负对数似然得到同一个 $\theta$。
(b) 证明 $H(p,q) = H(p) + D_{\mathrm{KL}}(p\,\|\,q)$，并说明为什么最小化交叉熵等价于最小化 KL 散度。
(c) 证明对词表大小为 $V$ 的均匀预测，困惑度等于 $V$。
(d) 某模型在验证集上的损失是每 token 3.0 nat。它的困惑度和每 token 比特数各是多少？

Close the derivation above and answer on paper.
(a) Show that maximizing $\prod_i p_\theta(x^{(i)})$ and minimizing the average negative log-likelihood give the same $\theta$.
(b) Show that $H(p,q) = H(p) + D_{\mathrm{KL}}(p\,\|\,q)$ and explain why minimizing cross-entropy is minimizing KL divergence.
(c) Show that a uniform prediction over a vocabulary of size $V$ has perplexity $V$.
(d) A model has a validation loss of 3.0 nats per token. What are its perplexity and its bits per token?

<details><summary><b>提示 1 / Hint 1</b></summary>

(a) 只需要两个事实，即 $\log$ 单调递增，以及乘以负常数会把最大化变成最小化。

(a) needs two facts. $\log$ is increasing, and multiplying by a negative constant turns a maximization into a minimization.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

(b) 在 $-\sum_x p\log q$ 中写 $\log q = \log p - \log\frac{p}{q}$。

(b) Inside $-\sum_x p\log q$ write $\log q = \log p - \log\frac{p}{q}$.

</details>

<details><summary><b>提示 3 / Hint 3</b></summary>

(d) 困惑度是 $e^{\mathcal L}$，比特数是 $\mathcal L/\ln 2$。

(d) Perplexity is $e^{\mathcal L}$ and bits are $\mathcal L/\ln 2$.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

(a) $\log$ 单调递增，所以 $\arg\max_\theta \prod_i p_\theta(x^{(i)}) = \arg\max_\theta \sum_i \log p_\theta(x^{(i)})$。乘以 $-\tfrac1N$ 后最大化变为最小化，位置不变，得到 $\arg\min_\theta -\tfrac1N\sum_i \log p_\theta(x^{(i)})$。

(b) $$H(p,q) = -\sum_x p(x)\log q(x) = -\sum_x p(x)\log p(x) + \sum_x p(x)\log\frac{p(x)}{q(x)} = H(p) + D_{\mathrm{KL}}(p\,\|\,q)$$ 其中 $p$ 是数据分布，$H(p)$ 不含 $\theta$，所以对 $\theta$ 最小化 $H(p, p_\theta)$ 与最小化 $D_{\mathrm{KL}}(p\,\|\,p_\theta)$ 相同。

(c) 每个位置 $p_\theta(x_t\mid x_{<t}) = 1/V$，所以 $\mathcal L = -\tfrac1T\sum_t \log\tfrac1V = \ln V$，$\mathrm{PPL} = e^{\ln V} = V$。

(d) $\mathrm{PPL} = e^{3.0} \approx 20.1$。比特数为 $3.0/\ln 2 \approx 4.33$ bit。也就是说模型平均相当于在约 20 个等可能的 token 中选择。

---

(a) $\log$ is increasing, so $\arg\max_\theta \prod_i p_\theta(x^{(i)}) = \arg\max_\theta \sum_i \log p_\theta(x^{(i)})$. Multiplying by $-\tfrac1N$ turns the maximization into a minimization at the same point, giving $\arg\min_\theta -\tfrac1N\sum_i \log p_\theta(x^{(i)})$.

(b) $$H(p,q) = -\sum_x p(x)\log q(x) = -\sum_x p(x)\log p(x) + \sum_x p(x)\log\frac{p(x)}{q(x)} = H(p) + D_{\mathrm{KL}}(p\,\|\,q)$$ Here $p$ is the data distribution and $H(p)$ does not contain $\theta$, so minimizing $H(p, p_\theta)$ over $\theta$ is the same as minimizing $D_{\mathrm{KL}}(p\,\|\,p_\theta)$.

(c) At every position $p_\theta(x_t\mid x_{<t}) = 1/V$, so $\mathcal L = -\tfrac1T\sum_t \log\tfrac1V = \ln V$ and $\mathrm{PPL} = e^{\ln V} = V$.

(d) $\mathrm{PPL} = e^{3.0} \approx 20.1$. Bits per token are $3.0/\ln 2 \approx 4.33$. On average the model is as uncertain as a uniform choice among about 20 tokens.

</details>